# Linear Regression from Scratch

Training a linear model using **gradient descent** — written from scratch, **no sklearn**.

Only used: `numpy` (math) and `matplotlib` (plot).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Proof this was built without sklearn
import sys
assert 'sklearn' not in sys.modules, 'sklearn was imported somewhere!'
print('sklearn is NOT imported (sys.modules clean) - this is from scratch')

## The Math

**Model:** $y = w \cdot x + b$

**Cost (MSE):** $J = \frac{1}{n}\sum (y_{pred} - y)^2$

**Gradients:**

$$\frac{\partial J}{\partial w} = \frac{2}{n} \sum x (y_{pred} - y)$$

$$\frac{\partial J}{\partial b} = \frac{2}{n} \sum (y_{pred} - y)$$

**Gradient descent update:**

$$w \leftarrow w - \alpha \frac{\partial J}{\partial w}, \quad b \leftarrow b - \alpha \frac{\partial J}{\partial b}$$

In [ ]:
class LinearRegressionScratch:
    def __init__(self, lr=0.01, epochs=5000):
        self.lr = lr
        self.epochs = epochs
        self.w = 0.0
        self.b = 0.0
        self.cost_hist = []
        self.w_hist = []
        self.b_hist = []

    def fit(self, X, y):
        n = len(X)
        for _ in range(self.epochs):
            y_pred = self.w * X + self.b
            error = y_pred - y

            dw = (2 / n) * np.sum(X * error)
            db = (2 / n) * np.sum(error)

            self.w -= self.lr * dw
            self.b -= self.lr * db

            self.cost_hist.append(np.mean(error ** 2))
            self.w_hist.append(self.w)
            self.b_hist.append(self.b)

        return self

    def predict(self, X):
        return self.w * X + self.b

## Data

Synthetic dataset generated with `numpy` — noise added on top of a line `y = 1.8x + 4`.

In [ ]:
rng = np.random.default_rng(42)
X = np.linspace(0, 10, 50)
y = 1.8 * X + 4 + rng.normal(0, 2, size=len(X))

print(f'Dataset: {len(X)} points. True line was y = 1.8x + 4')

## Training

Run gradient descent from scratch for 5000 epochs.

In [ ]:
model = LinearRegressionScratch(lr=0.01, epochs=5000)
model.fit(X, y)

print(f'Learned weight (slope):     {model.w:.4f}  (true: 1.8)')
print(f'Learned bias (intercept):   {model.b:.4f}  (true: 4.0)')

In [ ]:
y_pred = model.predict(X)

mse = np.mean((y_pred - y) ** 2)
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)
r2 = 1 - (ss_res / ss_tot)

print(f'MSE:  {mse:.4f}')
print(f'R2:   {r2:.4f}')

In [ ]:
# Export the static plots so they show on GitHub

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X, y, label='data', alpha=0.7)
axes[0].plot(X, y_pred, color='red', label=f'fit: y = {model.w:.2f}x + {model.b:.2f}')
axes[0].plot(X, 1.8 * X + 4, '--', color='gray', label='true: y = 1.8x + 4')
axes[0].set_title('Regression Fit (recovered vs true line)')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(model.cost_hist)
axes[1].set_title('Cost (MSE) over epochs')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('MSE')
axes[1].grid(alpha=0.3)

plt.tight_layout()
fig.savefig('fit.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Animate gradient descent converging, export as .gif so GitHub shows a movie

x_grid = np.linspace(0, 10, 200)
frames = np.linspace(0, len(model.w_hist) - 1, 60).astype(int)

fig, ax = plt.subplots(figsize=(7, 5))

def update(frame_idx):
    ax.clear()
    w = model.w_hist[frame_idx]
    b = model.b_hist[frame_idx]
    ax.scatter(X, y, alpha=0.6, label='data')
    ax.plot(x_grid, w * x_grid + b, 'r-', label=f'epoch {frame_idx:3d}: y = {w:.2f}x {b:+.2f}')
    ax.set_ylim(-5, 25)
    ax.set_title('Gradient Descent Learning the Line')
    ax.legend()
    ax.grid(alpha=0.3)
    return ax

anim = FuncAnimation(fig, update, frames=frames, repeat=True)
anim.save('training.gif', writer='pillow', fps=10, dpi=100)
plt.close(fig)

print('Saved fit.png and training.gif - download both into the repo folder')

## Result

The model recovered a slope near **1.8** and intercept near **4** from noisy data — gradient descent converged, cost decreased each epoch, and this all happens without a single sklearn import.